# MA-CMT Training — Google Colab Notebook

**Deep Learning-Based Missing-Aware Multi-Sensor Fusion for Agricultural Satellite Time-Series**

## Instructions
1. Set Runtime → Change runtime type → **T4 GPU**
2. Upload your dataset files to Google Drive:
   - `MyDrive/RioProject/data/raw/features.tar.gz`
   - `MyDrive/RioProject/data/raw/labels.geojson`
3. Run all cells in order.
4. Download the `.pt` checkpoints from `MyDrive/RioProject/checkpoints/`.
5. Place them in your local `RioProject/checkpoints/` directory.

In [ ]:
# ── 1. Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────────
!pip install -q torch numpy pandas scikit-learn h5py fastapi uvicorn

In [ ]:
# ── 3. Clone or copy project code ─────────────────────────────────────────────
# Option A: clone from GitHub (if you pushed the code)
# !git clone https://github.com/YOUR_USERNAME/RioProject.git /content/RioProject

# Option B: copy from Drive (if you uploaded the code)
import shutil, os
DRIVE_PROJECT = '/content/drive/MyDrive/RioProject'
LOCAL_PROJECT = '/content/RioProject'

if os.path.exists(DRIVE_PROJECT + '/ml'):
    shutil.copytree(DRIVE_PROJECT + '/ml',   LOCAL_PROJECT + '/ml',   dirs_exist_ok=True)
    shutil.copytree(DRIVE_PROJECT + '/data',  LOCAL_PROJECT + '/data', dirs_exist_ok=True)
    print('Copied from Drive')
else:
    print('WARNING: Code not found in Drive. Upload your RioProject folder to Google Drive first.')

In [ ]:
# ── 4. Verify GPU ─────────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ── 5. Load and cache the dataset ─────────────────────────────────────────────
import sys
sys.path.insert(0, '/content/RioProject')

from ml.dataset import load_all_samples, compute_normalization, normalize, stratified_split
import numpy as np

TAR_PATH = '/content/RioProject/data/raw/features.tar.gz'
GJ_PATH  = '/content/RioProject/data/raw/labels.geojson'
CACHE    = '/content/RioProject/data/processed/dataset_cache.pkl'
NORM_PATH = '/content/RioProject/data/processed/normalization.npz'

import os; os.makedirs('/content/RioProject/data/processed', exist_ok=True)
os.makedirs('/content/RioProject/checkpoints', exist_ok=True)

arrays, labels, ids = load_all_samples(TAR_PATH, GJ_PATH, cache_path=CACHE)
train_idx, val_idx, test_idx = stratified_split(arrays, labels, ids, seed=42)
print(f'Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}')

if os.path.exists(NORM_PATH):
    npz = np.load(NORM_PATH); mean, std = npz['mean'], npz['std']
else:
    mean, std = compute_normalization(arrays[train_idx])
    np.savez(NORM_PATH, mean=mean, std=std)

arrays_norm = normalize(arrays, mean, std)
print('Normalization done. Shape:', arrays_norm.shape)

In [ ]:
# ── 6. Training function ───────────────────────────────────────────────────────
import torch, torch.nn as nn, json
from torch.utils.data import DataLoader
from ml.dataset import CropDataset, compute_class_weights, CLASS_NAMES, N_CLASSES
from ml.models.model import build_model

def set_seed(s): 
    np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def train_model_colab(model_name, epochs=60, batch=256, lr=3e-4, patience=10, seed=42):
    set_seed(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'\n=== Training {model_name} on {device} ===')
    
    train_ds = CropDataset(arrays_norm[train_idx], labels[train_idx])
    val_ds   = CropDataset(arrays_norm[val_idx],   labels[val_idx])
    test_ds  = CropDataset(arrays_norm[test_idx],  labels[test_idx])
    
    # num_workers=2 on Colab (Linux)
    train_loader = DataLoader(train_ds, batch_size=batch, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch, shuffle=False, num_workers=2)
    test_loader  = DataLoader(test_ds,  batch_size=batch, shuffle=False, num_workers=2)
    
    model = build_model(model_name, n_classes=N_CLASSES).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Parameters: {n_params:,}')
    
    cw = compute_class_weights(labels[train_idx], N_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss(weight=cw)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda')
    
    best_val_acc, patience_ctr = -1.0, 0
    ckpt_path = f'/content/RioProject/checkpoints/{model_name}.pt'
    
    for epoch in range(1, epochs + 1):
        # Train
        model.train(); tr_loss = tr_correct = tr_total = 0
        for s1, s2, m1, m2, y in train_loader:
            s1,s2,m1,m2,y = s1.to(device),s2.to(device),m1.to(device),m2.to(device),y.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                out = model(s1,s2,m1,m2); loss = criterion(out, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            tr_loss += loss.item()*len(y); tr_correct += (out.argmax(1)==y).sum().item(); tr_total += len(y)
        
        # Validate
        model.eval(); va_correct = va_total = 0
        with torch.no_grad():
            for s1,s2,m1,m2,y in val_loader:
                s1,s2,m1,m2 = s1.to(device),s2.to(device),m1.to(device),m2.to(device)
                out = model(s1,s2,m1,m2)
                va_correct += (out.argmax(1)==y.to(device)).sum().item(); va_total += len(y)
        
        va_acc = va_correct / va_total
        scheduler.step()
        
        if epoch % 5 == 0 or epoch == 1:
            print(f'Ep {epoch:03d} | tr_acc={tr_correct/tr_total:.4f} | val_acc={va_acc:.4f}')
        
        if va_acc > best_val_acc:
            best_val_acc = va_acc; patience_ctr = 0
            torch.save({'model_name': model_name, 'state_dict': model.state_dict(),
                        'epoch': epoch, 'val_acc': va_acc,
                        'mean': mean.tolist(), 'std': std.tolist(),
                        'class_names': CLASS_NAMES}, ckpt_path)
        else:
            patience_ctr += 1
            if patience_ctr >= patience: print(f'Early stop at epoch {epoch}'); break
    
    # Test
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['state_dict']); model.eval()
    te_correct = te_total = 0
    with torch.no_grad():
        for s1,s2,m1,m2,y in test_loader:
            s1,s2,m1,m2 = s1.to(device),s2.to(device),m1.to(device),m2.to(device)
            out = model(s1,s2,m1,m2)
            te_correct += (out.argmax(1)==y.to(device)).sum().item(); te_total += len(y)
    print(f'{model_name} → Best val: {best_val_acc:.4f} | Test: {te_correct/te_total:.4f}')
    return best_val_acc, te_correct/te_total

print('Training function defined. Ready to train.')

In [ ]:
# ── 7. Train all models ────────────────────────────────────────────────────────
# Expected time on T4: ~8–15 min per model (60 epochs, 42k samples)
results_summary = {}
for model_name in ['macmt', 's1only', 's2only', 'concat']:
    val_acc, test_acc = train_model_colab(model_name, epochs=60, batch=256, patience=10)
    results_summary[model_name] = {'val_acc': val_acc, 'test_acc': test_acc}

print('\n=== Training Summary ===')
for m, r in results_summary.items():
    print(f'{m:8s} val={r["val_acc"]:.4f}  test={r["test_acc"]:.4f}')

In [ ]:
# ── 8. Run experiments ─────────────────────────────────────────────────────────
sys.path.insert(0, '/content/RioProject')
import importlib, ml.experiments.run_experiments as run_exp
importlib.reload(run_exp)

# Override paths for Colab
from pathlib import Path
run_exp.DATA_DIR  = Path('/content/RioProject/data/raw')
run_exp.PROC_DIR  = Path('/content/RioProject/data/processed')
run_exp.CKPT_DIR  = Path('/content/RioProject/checkpoints')
run_exp.RES_DIR   = Path('/content/RioProject/results')
run_exp.RES_DIR.mkdir(parents=True, exist_ok=True)

all_results = run_exp.run_experiments()
print('Experiments complete.')

In [ ]:
# ── 9. Copy outputs back to Drive ─────────────────────────────────────────────
import shutil
DRIVE_OUT = '/content/drive/MyDrive/RioProject'

shutil.copytree('/content/RioProject/checkpoints', DRIVE_OUT + '/checkpoints', dirs_exist_ok=True)
shutil.copytree('/content/RioProject/results',     DRIVE_OUT + '/results',     dirs_exist_ok=True)
shutil.copytree('/content/RioProject/data/processed', DRIVE_OUT + '/data/processed', dirs_exist_ok=True)
print('Copied checkpoints and results to Drive.')
print('Download from:', DRIVE_OUT + '/checkpoints')

In [ ]:
# ── 10. Quick accuracy vs missing-rate plot ───────────────────────────────────
import json, matplotlib.pyplot as plt

with open('/content/RioProject/results/experiment_results.json') as f:
    exp = json.load(f)

models  = ['macmt', 's1only', 's2only', 'concat']
labels_ = ['MA-CMT', 'S1-Only', 'S2-Only', 'Concat']
colors  = ['#166534','#1d4ed8','#d97706','#7c3aed']
rates   = [0, 10, 20, 30, 50]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
scenarios = [('random','Random Missing'),('s1_missing','S1 Missing'),
             ('s2_missing','S2 Missing'),('consecutive','Consecutive')]

for ax, (scenario, title) in zip(axes.flat, scenarios):
    for model, label, color in zip(models, labels_, colors):
        accs = []
        for rate in rates:
            if rate == 0:
                d = exp.get('complete', {}).get(model)
                accs.append(d['accuracy'] * 100 if d else None)
            else:
                d = exp.get(scenario, {}).get(model, {})
                m = d.get(str(rate)) if d else None
                accs.append(m['accuracy'] * 100 if m else None)
        ax.plot(rates, accs, marker='o', label=label, color=color, linewidth=2)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Missing Rate (%)')
    ax.set_ylabel('Accuracy (%)')
    ax.set_ylim(0, 100); ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/RioProject/results/accuracy_vs_missing.png', dpi=150, bbox_inches='tight')
plt.savefig(DRIVE_OUT + '/results/accuracy_vs_missing.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')